In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import matplotlib.pyplot as plt
import numpy as np

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print("Ready!")

c:\Users\devuser3\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6471.07it/s]


Ready!


In [2]:
# Test text
text = "The quick brown fox jumps over the lazy dog and then runs away quickly"

inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0]  # shape: [n_tokens, 50257]


print(f"Text: {text}\n")
print(f"{'Position':<10} {'Actual':<12} {'Predicted':<12} {'Correct':<8} {'Prob':<8}")
print("-"*55)

correct_count = 0
total = len(tokens) - 1  

for i in range(total):
    actual_token    = tokens[i+1]
    actual_id       = inputs['input_ids'][0][i+1].item()
    
    probs           = torch.softmax(logits[i], dim=0)
    predicted_id    = probs.argmax().item()
    predicted_token = tokenizer.decode([predicted_id])
    
    prob_of_actual  = probs[actual_id].item()
    is_correct      = predicted_id == actual_id
    
    if is_correct:
        correct_count += 1
    
    mark = "✅" if is_correct else "❌"
    print(f"{i:<10} {actual_token:<12} {predicted_token:<12} {mark:<8} {prob_of_actual:.4f}")

print(f"\nAccuracy: {correct_count}/{total} = {correct_count/total*100:.1f}%")

Text: The quick brown fox jumps over the lazy dog and then runs away quickly

Position   Actual       Predicted    Correct  Prob    
-------------------------------------------------------
0          Ġquick       
            ❌        0.0002
1          Ġbrown       -            ❌        0.0001
2          Ġfox         ie           ❌        0.0019
3          Ġjumps       es           ❌        0.0078
4          Ġover         up          ❌        0.0641
5          Ġthe          the         ✅        0.4228
6          Ġlazy         fence       ❌        0.0007
7          Ġdog         ,            ❌        0.0220
8          Ġand          and         ✅        0.3205
9          Ġthen         runs        ❌        0.0068
10         Ġruns         runs        ✅        0.0291
11         Ġaway         off         ❌        0.1023
12         Ġquickly     .            ❌        0.0015

Accuracy: 3/13 = 23.1%


In [3]:
test_texts = [
    "The cat sat on the mat and looked around",
    "Python is a popular programming language used for",
    "Once upon a time there was a beautiful princess who",
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits[0]
    correct = 0
    total   = len(tokens) - 1
    
    for i in range(total):
        actual_id    = inputs['input_ids'][0][i+1].item()
        probs        = torch.softmax(logits[i], dim=0)
        predicted_id = probs.argmax().item()
        if predicted_id == actual_id:
            correct += 1
    
    print(f"Text: {text[:40]}...")
    print(f"Accuracy: {correct}/{total} = {correct/total*100:.1f}%\n")

Text: The cat sat on the mat and looked around...
Accuracy: 3/8 = 37.5%

Text: Python is a popular programming language...
Accuracy: 3/7 = 42.9%

Text: Once upon a time there was a beautiful p...
Accuracy: 4/9 = 44.4%



* GPT-2 achieves around **~23% to 44% accuracy** depending on the type of text.

* **Common phrases** tend to be **more accurate**, since the model has seen them frequently during training.

* Compared to a random guess, GPT-2 is **~1000× better**, showing strong pattern learning.

* Frequent words like **“the”** and **“and”** are **much easier to predict** due to their high occurrence in language.
